In [160]:
# Selecciona Income y Período para descargar: 
IncomeFile = "Income_2025.csv"
Periodo = 'Anual' # 'Anual' si es Resultado del Año. 'Q1'..'Q4' si es Trimestral.

### Tranformación Automatizada Datos Holded:

In [161]:
import pandas as pd 
import numpy as np 

In [162]:
# Importamos data:
    # IncomeFile = "Ingresos2022.csv"
    # Periodo = 'Anual'

Ingresos = pd.read_csv(f"C:/Users/Jorge Lozano/Desktop/GitHub_Holded/Data/Income/{IncomeFile}", header = None, sep = ',') 

In [163]:
# Renombramos nombres de columnas:
Ingresos = Ingresos.rename(columns={0 : 'Subcuenta', 1 : 'Valor'})

In [164]:
Ingresos = Ingresos.iloc[5:].reset_index(drop=True)

* 0. Cabecera y pie de página:

In [165]:
# Borramos filas totalmente vacías (relleno del export de Holded):
Ingresos = Ingresos.dropna(subset=['Subcuenta', 'Valor'], how='all')

In [166]:
Ingresos

,Subcuenta,Valor,2
0,1. Importe neto de la cifra de negocios,"3.824.116,90",NaN
1,b) Prestaciones de servicios,"3.824.116,90",NaN
2,71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA,"3.817.233,49",NaN
3,71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA,"6.883,41",NaN
5,2. Variación de existencias de productos termi...,"27.151,23",NaN
...,...,...,...
116,19. Impuestos sobre beneficios,"-119.494,40",NaN
117,Impuestos sobre beneficios,"-119.494,40",NaN
118,63010000 - IMPUESTO CORRIENTE,"-119.494,40",NaN
120,Resultado del ejercicio,"496.456,12",NaN


In [167]:
# Textos fijos de cabecera y pie que trae siempre el informe de Holded:
Textos_informe = pd.Series(['Tet¡xto', 'Informe creado automáticamente'])
print(Textos_informe)

0                           Tet¡xto
1    Informe creado automáticamente
dtype: object


In [168]:
# Borramos cabecera (título, rango de fechas) y pie de página:
Ingresos = Ingresos[~Ingresos['Subcuenta'].str.contains('|'.join(Textos_informe), na=False)]
Ingresos = Ingresos[~Ingresos['Subcuenta'].str.match(r'^\d{2}/\d{2}/\d{4}', na=False)]
Ingresos = Ingresos[Ingresos['Valor'] != 'Total']

In [169]:
Ingresos

,Subcuenta,Valor,2
0,1. Importe neto de la cifra de negocios,"3.824.116,90",NaN
1,b) Prestaciones de servicios,"3.824.116,90",NaN
2,71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA,"3.817.233,49",NaN
3,71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA,"6.883,41",NaN
5,2. Variación de existencias de productos termi...,"27.151,23",NaN
...,...,...,...
114,Resultado antes de impuestos,"615.950,52",NaN
116,19. Impuestos sobre beneficios,"-119.494,40",NaN
117,Impuestos sobre beneficios,"-119.494,40",NaN
118,63010000 - IMPUESTO CORRIENTE,"-119.494,40",NaN


* 1. Resultados (subtotales, no forman parte de la jerarquía Grupo/Subgrupo/Cuenta):

In [170]:
Resultados = pd.Series(['Resultado de explotación', 'Resultado financiero',
                         'Resultado antes de impuestos', 'Resultado del ejercicio'])
print(Resultados)

0        Resultado de explotación
1            Resultado financiero
2    Resultado antes de impuestos
3         Resultado del ejercicio
dtype: object


In [171]:
# Borramos las filas de Resultado:
Ingresos = Ingresos[~Ingresos['Subcuenta'].isin(Resultados)]
# Borramos también las filas separadoras (Subcuenta vacía, Valor = 0):
Ingresos = Ingresos[~Ingresos['Subcuenta'].isna()]

In [172]:
Ingresos

,Subcuenta,Valor,2
0,1. Importe neto de la cifra de negocios,"3.824.116,90",NaN
1,b) Prestaciones de servicios,"3.824.116,90",NaN
2,71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA,"3.817.233,49",NaN
3,71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA,"6.883,41",NaN
5,2. Variación de existencias de productos termi...,"27.151,23",NaN
...,...,...,...
109,b) Resultados por enajenaciones y otras,"17.973,35",NaN
110,76610000 - BENEFICIO VALORES REPRESENTATIVOS D...,"17.973,35",NaN
116,19. Impuestos sobre beneficios,"-119.494,40",NaN
117,Impuestos sobre beneficios,"-119.494,40",NaN


* 2. Grupo:

In [173]:
Grupo_id = pd.Series(['1.', '2.', '3.', '4.', '5.', '6.', '7.', '8.', '9.', '10.',
                       '11.', '12.', '13.', '14.', '15.', '16.', '17.', '18.', '19.', '20.'])
print(Grupo_id)

0      1.
1      2.
2      3.
3      4.
4      5.
5      6.
6      7.
7      8.
8      9.
9     10.
10    11.
11    12.
12    13.
13    14.
14    15.
15    16.
16    17.
17    18.
18    19.
19    20.
dtype: object


In [174]:
Grupo_id = Grupo_id.tolist()

In [175]:
# Para crear columna Grupo:
Ingresos['Grupo'] = Ingresos['Subcuenta'].where(Ingresos['Subcuenta'].str.startswith(tuple(Grupo_id))).ffill()
# Para Borrar Filas Grupo Original:
Ingresos = Ingresos[~Ingresos['Subcuenta'].str.startswith(tuple(Grupo_id))]

In [176]:
Ingresos

,Subcuenta,Valor,2,Grupo
1,b) Prestaciones de servicios,"3.824.116,90",NaN,1. Importe neto de la cifra de negocios
2,71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA,"3.817.233,49",NaN,1. Importe neto de la cifra de negocios
3,71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA,"6.883,41",NaN,1. Importe neto de la cifra de negocios
6,Variación de existencias de productos terminad...,"27.151,23",NaN,2. Variación de existencias de productos termi...
7,79310000 - REVERSION VARIACION PROD TERM Y CUR...,"27.151,23",NaN,2. Variación de existencias de productos termi...
...,...,...,...,...
106,66310000 - PERDIDAS DE CARTERA DE NEGOCIACION,"-114,72",NaN,16. Variación de valor razonable en instrument...
109,b) Resultados por enajenaciones y otras,"17.973,35",NaN,18. Deterioro y resultado por enajenaciones de...
110,76610000 - BENEFICIO VALORES REPRESENTATIVOS D...,"17.973,35",NaN,18. Deterioro y resultado por enajenaciones de...
117,Impuestos sobre beneficios,"-119.494,40",NaN,19. Impuestos sobre beneficios


* Subgrupo:

In [177]:
# Holded no antepone letra cuando el Grupo solo tiene un único Subgrupo (ej: "Amortización 
# del inmovilizado", "Diferencias de cambio"). Normalizamos esas filas añadiendo "a) " 
# para que sigan el mismo patrón que el resto:
Subgrupos_sin_letra = pd.Series(['Amortización del inmovilizado', 'Diferencias de cambio',
                                  'Impuestos sobre beneficios'])
print(Subgrupos_sin_letra)

0    Amortización del inmovilizado
1            Diferencias de cambio
2       Impuestos sobre beneficios
dtype: object


In [178]:
Ingresos['Subcuenta'] = Ingresos['Subcuenta'].mask(Ingresos['Subcuenta'].isin(Subgrupos_sin_letra),
                                                     'a) ' + Ingresos['Subcuenta'])

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_17672\1586696080.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Ingresos['Subcuenta'] = Ingresos['Subcuenta'].mask(Ingresos['Subcuenta'].isin(Subgrupos_sin_letra),


In [179]:
Ingresos

,Subcuenta,Valor,2,Grupo
1,b) Prestaciones de servicios,"3.824.116,90",NaN,1. Importe neto de la cifra de negocios
2,71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA,"3.817.233,49",NaN,1. Importe neto de la cifra de negocios
3,71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA,"6.883,41",NaN,1. Importe neto de la cifra de negocios
6,Variación de existencias de productos terminad...,"27.151,23",NaN,2. Variación de existencias de productos termi...
7,79310000 - REVERSION VARIACION PROD TERM Y CUR...,"27.151,23",NaN,2. Variación de existencias de productos termi...
...,...,...,...,...
106,66310000 - PERDIDAS DE CARTERA DE NEGOCIACION,"-114,72",NaN,16. Variación de valor razonable en instrument...
109,b) Resultados por enajenaciones y otras,"17.973,35",NaN,18. Deterioro y resultado por enajenaciones de...
110,76610000 - BENEFICIO VALORES REPRESENTATIVOS D...,"17.973,35",NaN,18. Deterioro y resultado por enajenaciones de...
117,a) Impuestos sobre beneficios,"-119.494,40",NaN,19. Impuestos sobre beneficios


In [180]:
Subgrupo_id = pd.Series(['a)', 'b)', 'c)', 'd)', 'e)', 'f)', 'g)']).tolist()

In [181]:
# Creamos columna Subgrupo:
Ingresos['SubGrupo'] = Ingresos['Subcuenta'].where(Ingresos['Subcuenta'].str.startswith(tuple(Subgrupo_id))).ffill()
# Eliminamos columna original con Subgrupo:
Ingresos = Ingresos[~Ingresos['Subcuenta'].str.startswith(tuple(Subgrupo_id))]

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_17672\1014142293.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Ingresos['SubGrupo'] = Ingresos['Subcuenta'].where(Ingresos['Subcuenta'].str.startswith(tuple(Subgrupo_id))).ffill()


In [182]:
Ingresos

,Subcuenta,Valor,2,Grupo,SubGrupo
2,71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA,"3.817.233,49",NaN,1. Importe neto de la cifra de negocios,b) Prestaciones de servicios
3,71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA,"6.883,41",NaN,1. Importe neto de la cifra de negocios,b) Prestaciones de servicios
6,Variación de existencias de productos terminad...,"27.151,23",NaN,2. Variación de existencias de productos termi...,b) Prestaciones de servicios
7,79310000 - REVERSION VARIACION PROD TERM Y CUR...,"27.151,23",NaN,2. Variación de existencias de productos termi...,b) Prestaciones de servicios
11,61100000 - TRABAJOS SUBCONTRATADOS DIVERSOS,"-160.987,67",NaN,4. Aprovisionamientos,c) Trabajos realizados por otras empresas
...,...,...,...,...,...
101,66240001 - PRESTAMO BANCO SOLVIA,"-592,74",NaN,15. Gastos financieros,b) Por deudas con terceros
102,66900002 - PERDIDAS OPERACIONES CON USUARIOS,"-37.648,43",NaN,15. Gastos financieros,b) Por deudas con terceros
106,66310000 - PERDIDAS DE CARTERA DE NEGOCIACION,"-114,72",NaN,16. Variación de valor razonable en instrument...,a) Cartera de negociación y otros
110,76610000 - BENEFICIO VALORES REPRESENTATIVOS D...,"17.973,35",NaN,18. Deterioro y resultado por enajenaciones de...,b) Resultados por enajenaciones y otras


* Cuenta: lo que queda son las cuentas hoja (código + " - " + descripción)

In [183]:
print(Ingresos)

                                             Subcuenta         Valor   2  \
2     71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA  3.817.233,49 NaN   
3      71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA      6.883,41 NaN   
6    Variación de existencias de productos terminad...     27.151,23 NaN   
7    79310000 - REVERSION VARIACION PROD TERM Y CUR...     27.151,23 NaN   
11         61100000 - TRABAJOS SUBCONTRATADOS DIVERSOS   -160.987,67 NaN   
..                                                 ...           ...  ..   
101                   66240001 - PRESTAMO BANCO SOLVIA       -592,74 NaN   
102       66900002 - PERDIDAS OPERACIONES CON USUARIOS    -37.648,43 NaN   
106      66310000 - PERDIDAS DE CARTERA DE NEGOCIACION       -114,72 NaN   
110  76610000 - BENEFICIO VALORES REPRESENTATIVOS D...     17.973,35 NaN   
118                      63010000 - IMPUESTO CORRIENTE   -119.494,40 NaN   

                                                 Grupo  \
2              1. Importe net

In [184]:
# Configuración Año y Periodo (Establecidos en el inicio del script)

Año = IncomeFile[7:11]

Ingresos['Año'] = Año
Ingresos['Periodo'] = Periodo

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_17672\3393028350.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Ingresos['Año'] = Año
C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_17672\3393028350.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Ingresos['Periodo'] = Periodo


* Ordenar Columnas:

In [185]:
Ingresos = Ingresos[['Año', 'Periodo', 'Grupo', 'SubGrupo', 'Subcuenta', 'Valor']]

In [186]:
Ingresos

,Año,Periodo,Grupo,SubGrupo,Subcuenta,Valor
2,2025,Anual,1. Importe neto de la cifra de negocios,b) Prestaciones de servicios,71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA,"3.817.233,49"
3,2025,Anual,1. Importe neto de la cifra de negocios,b) Prestaciones de servicios,71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA,"6.883,41"
6,2025,Anual,2. Variación de existencias de productos termi...,b) Prestaciones de servicios,Variación de existencias de productos terminad...,"27.151,23"
7,2025,Anual,2. Variación de existencias de productos termi...,b) Prestaciones de servicios,79310000 - REVERSION VARIACION PROD TERM Y CUR...,"27.151,23"
11,2025,Anual,4. Aprovisionamientos,c) Trabajos realizados por otras empresas,61100000 - TRABAJOS SUBCONTRATADOS DIVERSOS,"-160.987,67"
...,...,...,...,...,...,...
101,2025,Anual,15. Gastos financieros,b) Por deudas con terceros,66240001 - PRESTAMO BANCO SOLVIA,"-592,74"
102,2025,Anual,15. Gastos financieros,b) Por deudas con terceros,66900002 - PERDIDAS OPERACIONES CON USUARIOS,"-37.648,43"
106,2025,Anual,16. Variación de valor razonable en instrument...,a) Cartera de negociación y otros,66310000 - PERDIDAS DE CARTERA DE NEGOCIACION,"-114,72"
110,2025,Anual,18. Deterioro y resultado por enajenaciones de...,b) Resultados por enajenaciones y otras,76610000 - BENEFICIO VALORES REPRESENTATIVOS D...,"17.973,35"


* Pendiente Transformación: Desglosar Columnas correspondientes en {Id y Nombre}

In [187]:
# Division Columnas y creación columnas_id's: 

Ingresos[['Grupo_id', 'Grupo']] = Ingresos['Grupo'].str.split('. ', n = 1, expand = True)         # Grupo
Ingresos[['SubGrupo_id', 'SubGrupo']] = Ingresos['SubGrupo'].str.split(') ', n = 1, expand = True, regex = False) # SubGrupo
Ingresos[['Cuenta_id', 'Cuenta']] = Ingresos['Subcuenta'].str.split(' - ', n = 1, expand = True)   # Cuenta

In [188]:
Ingresos.head(5)

,Año,Periodo,Grupo,SubGrupo,Subcuenta,Valor,Grupo_id,SubGrupo_id,Cuenta_id,Cuenta
2,2025,Anual,Importe neto de la cifra de negocios,Prestaciones de servicios,71100000 - INGRESOS POR SERVICIOS DE CONSULTORIA,"3.817.233,49",1,b,71100000,INGRESOS POR SERVICIOS DE CONSULTORIA
3,2025,Anual,Importe neto de la cifra de negocios,Prestaciones de servicios,71100001 - INGRESOS POR LICENCIAS DE PLATAFORMA,"6.883,41",1,b,71100001,INGRESOS POR LICENCIAS DE PLATAFORMA
6,2025,Anual,Variación de existencias de productos terminad...,Prestaciones de servicios,Variación de existencias de productos terminad...,"27.151,23",2,b,Variación de existencias de productos terminad...,None
7,2025,Anual,Variación de existencias de productos terminad...,Prestaciones de servicios,79310000 - REVERSION VARIACION PROD TERM Y CUR...,"27.151,23",2,b,79310000,REVERSION VARIACION PROD TERM Y CUR FAB
11,2025,Anual,Aprovisionamientos,Trabajos realizados por otras empresas,61100000 - TRABAJOS SUBCONTRATADOS DIVERSOS,"-160.987,67",4,c,61100000,TRABAJOS SUBCONTRATADOS DIVERSOS


In [189]:
Ingresos.columns

Index(['Año', 'Periodo', 'Grupo', 'SubGrupo', 'Subcuenta', 'Valor', 'Grupo_id',
       'SubGrupo_id', 'Cuenta_id', 'Cuenta'],
      dtype='object')

In [190]:
# Ordenamos el dataframe: 
Ingresos = Ingresos[['Año', 'Grupo_id', 'Grupo', 'SubGrupo_id', 'SubGrupo',
                      'Cuenta_id', 'Cuenta', 'Valor']]

In [191]:
# Cambiamos tipo de dato de Valor de Object a Numeric:
Ingresos['Valor'] = Ingresos['Valor'].str.replace('.', '', regex=False)   # Elimina los puntos que separan miles
Ingresos['Valor'] = Ingresos['Valor'].str.replace(',', '.', regex=False)  # Reemplaza las comas por puntos decimales
Ingresos['Valor'] = pd.to_numeric(Ingresos['Valor'], errors='coerce')

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_17672\2783116765.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Ingresos['Valor'] = Ingresos['Valor'].str.replace('.', '', regex=False)   # Elimina los puntos que separan miles
C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_17672\2783116765.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Ingresos['Valor'] = Ingresos['Valor'].str.replace(',', '.', regex=False)  # Reemplaza las comas por puntos decimales
C:\Users\Jorge Lozano\AppData\Local\Temp\i

In [192]:
# Creamos columna Valores Absolutos:
Ingresos['Abs Valor'] = Ingresos['Valor'].abs()

C:\Users\Jorge Lozano\AppData\Local\Temp\ipykernel_17672\2539220415.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Ingresos['Abs Valor'] = Ingresos['Valor'].abs()


In [193]:
# Renombramos columnas finales para que coincidan con el resto de reportes:
Ingresos = Ingresos.rename(columns={'Grupo_id': 'ID Grupo', 'SubGrupo_id': 'ID Subgrupo',
                                     'Cuenta_id': 'ID Cuenta', 'Valor': 'Valores', 'Abs Valor': 'Abs Valores'})
Ingresos['ID Grupo'] = pd.to_numeric(Ingresos['ID Grupo'])

In [194]:
Ingresos.head(10)

,Año,ID Grupo,Grupo,ID Subgrupo,SubGrupo,ID Cuenta,Cuenta,Valores,Abs Valores
2,2025,1,Importe neto de la cifra de negocios,b,Prestaciones de servicios,71100000,INGRESOS POR SERVICIOS DE CONSULTORIA,3817233.49,3817233.49
3,2025,1,Importe neto de la cifra de negocios,b,Prestaciones de servicios,71100001,INGRESOS POR LICENCIAS DE PLATAFORMA,6883.41,6883.41
6,2025,2,Variación de existencias de productos terminad...,b,Prestaciones de servicios,Variación de existencias de productos terminad...,None,27151.23,27151.23
7,2025,2,Variación de existencias de productos terminad...,b,Prestaciones de servicios,79310000,REVERSION VARIACION PROD TERM Y CUR FAB,27151.23,27151.23
11,2025,4,Aprovisionamientos,c,Trabajos realizados por otras empresas,61100000,TRABAJOS SUBCONTRATADOS DIVERSOS,-160987.67,160987.67
12,2025,4,Aprovisionamientos,c,Trabajos realizados por otras empresas,61100001,COMPASS FIELDING NETWORK SERVICES,-1473784.06,1473784.06
13,2025,4,Aprovisionamientos,c,Trabajos realizados por otras empresas,61100002,ESTUARY PAYMENT SOLUTIONS,-24895.00,24895.00
17,2025,6,Gastos de personal,a,Sueldos salarios y asimilados,64010000,SUELDOS Y SALARIOS,-563001.78,563001.78
19,2025,6,Gastos de personal,b,Cargos sociales,64210000,SEGURIDAD SOCIAL A CARGO DE LA EMPRESA,-110042.79,110042.79
23,2025,7,Otros gastos de explotación,a,Servicios exteriores,62100000,ARRENDAMIENTOS Y CANONES,-4808.94,4808.94


### Resultado y Descarga: 

In [195]:
Ingresos.head(4)

,Año,ID Grupo,Grupo,ID Subgrupo,SubGrupo,ID Cuenta,Cuenta,Valores,Abs Valores
2,2025,1,Importe neto de la cifra de negocios,b,Prestaciones de servicios,71100000,INGRESOS POR SERVICIOS DE CONSULTORIA,3817233.49,3817233.49
3,2025,1,Importe neto de la cifra de negocios,b,Prestaciones de servicios,71100001,INGRESOS POR LICENCIAS DE PLATAFORMA,6883.41,6883.41
6,2025,2,Variación de existencias de productos terminad...,b,Prestaciones de servicios,Variación de existencias de productos terminad...,None,27151.23,27151.23
7,2025,2,Variación de existencias de productos terminad...,b,Prestaciones de servicios,79310000,REVERSION VARIACION PROD TERM Y CUR FAB,27151.23,27151.23


In [196]:
### Descarga arhivo: 
NombreArchivo = f"File.Income{Año}.csv"
Ingresos.to_csv(NombreArchivo, index=False)

In [197]:
f"Archivo creado llamado: 'File.Income{Año}.csv'"

"Archivo creado llamado: 'File.Income2025.csv'"